In [11]:
import pandas as pd
import json

# 1. Load the CSV file
df = pd.read_csv('ufc_fight_data.csv')

pd.set_option('display.max_columns', None)


In [12]:
# 2. Define the parsing function
def parse_fight_details(row):
    try:
        # Parse the JSON string
        details = json.loads(row['details_json'])
        
        f1_name = row['fighter_1']
        f2_name = row['fighter_2']
        
        flat_data = {}
        
        # --- A. Extract Meta Data ---
        if 'meta' in details:
            for k, v in details['meta'].items():
                flat_data[f'meta_{k}'] = v
        
        # --- Helper: Extract specific dictionary ---
        # This reduces code repetition. It takes a data dictionary (like a specific round)
        # and extracts the stats for both fighter 1 and fighter 2.
        def extract_from_dict(source_dict, prefix_f1, prefix_f2):
            # Fighter 1 stats
            if f1_name in source_dict:
                for k, v in source_dict[f1_name].items():
                    flat_data[f'{prefix_f1}_{k}'] = v
            
            # Fighter 2 stats
            if f2_name in source_dict:
                for k, v in source_dict[f2_name].items():
                    flat_data[f'{prefix_f2}_{k}'] = v

        # --- B. Match Totals (The Aggregate) ---
        if 'totals' in details and 'match' in details['totals']:
            extract_from_dict(
                details['totals']['match'], 
                'fighter_1_match', 
                'fighter_2_match'
            )

        # --- C. Significant Strikes Match Totals ---
        if 'significant_strikes' in details and 'match' in details['significant_strikes']:
            extract_from_dict(
                details['significant_strikes']['match'], 
                'fighter_1_sig_str', 
                'fighter_2_sig_str'
            )

        # --- D. ROUND-BY-ROUND Totals ---
        # We iterate through the list of rounds provided in the JSON.
        # This dynamically creates keys like "round_1_fighter_1_kd", "round_2_fighter_1_kd", etc.
        if 'totals' in details and 'rounds' in details['totals']:
            for r_data in details['totals']['rounds']:
                r_num = r_data.get('round')
                extract_from_dict(
                    r_data, 
                    f'round_{r_num}_fighter_1_total', 
                    f'round_{r_num}_fighter_2_total'
                )

        # --- E. ROUND-BY-ROUND Sig Strikes ---
        if 'significant_strikes' in details and 'rounds' in details['significant_strikes']:
            for r_data in details['significant_strikes']['rounds']:
                r_num = r_data.get('round')
                extract_from_dict(
                    r_data, 
                    f'round_{r_num}_fighter_1_sig_str', 
                    f'round_{r_num}_fighter_2_sig_str'
                )

        return pd.Series(flat_data)
        
    except Exception as e:
        # Return empty if parsing fails
        return pd.Series({})

# 3. Apply the function
print("Parsing JSON columns...")
expanded_columns = df.apply(parse_fight_details, axis=1)

# 4. Concatenate
df_formatted = pd.concat([df.drop(columns=['details_json', 'fight_url']), expanded_columns], axis=1)

# 5. Optional: Reorder columns alphabetically so Round 1, Round 2, etc. stay together
df_formatted = df_formatted.reindex(sorted(df_formatted.columns), axis=1)


df_formatted.head()

Parsing JSON columns...


,event_date,fighter_1,fighter_1_match_ctrl,fighter_1_match_kd,fighter_1_match_rev,fighter_1_match_sig_str,fighter_1_match_sig_str_prcnt,fighter_1_match_sub_att,fighter_1_match_td,fighter_1_match_td_prcnt,fighter_1_match_total_str,fighter_1_sig_str_body,fighter_1_sig_str_clinch,fighter_1_sig_str_distance,fighter_1_sig_str_ground,fighter_1_sig_str_head,fighter_1_sig_str_leg,fighter_1_sig_str_sig_str,fighter_1_sig_str_sig_str_prcnt,fighter_2,fighter_2_match_ctrl,fighter_2_match_kd,fighter_2_match_rev,fighter_2_match_sig_str,fighter_2_match_sig_str_prcnt,fighter_2_match_sub_att,fighter_2_match_td,fighter_2_match_td_prcnt,fighter_2_match_total_str,fighter_2_sig_str_body,fighter_2_sig_str_clinch,fighter_2_sig_str_distance,fighter_2_sig_str_ground,fighter_2_sig_str_head,fighter_2_sig_str_leg,fighter_2_sig_str_sig_str,fighter_2_sig_str_sig_str_prcnt,meta_first_fighter_won,meta_method,meta_referee,meta_round,meta_time,meta_time_format,method,round,round_1_fighter_1_sig_str_body,round_1_fighter_1_sig_str_clinch,round_1_fighter_1_sig_str_distance,round_1_fighter_1_sig_str_ground,round_1_fighter_1_sig_str_head,round_1_fighter_1_sig_str_leg,round_1_fighter_1_sig_str_sig_str,round_1_fighter_1_sig_str_sig_str_prcnt,round_1_fighter_1_total_ctrl,round_1_fighter_1_total_kd,round_1_fighter_1_total_rev,round_1_fighter_1_total_sig_str,round_1_fighter_1_total_sig_str_prcnt,round_1_fighter_1_total_sub_att,round_1_fighter_1_total_td,round_1_fighter_1_total_td_prcnt,round_1_fighter_1_total_total_str,round_1_fighter_2_sig_str_body,round_1_fighter_2_sig_str_clinch,round_1_fighter_2_sig_str_distance,round_1_fighter_2_sig_str_ground,round_1_fighter_2_sig_str_head,round_1_fighter_2_sig_str_leg,round_1_fighter_2_sig_str_sig_str,round_1_fighter_2_sig_str_sig_str_prcnt,round_1_fighter_2_total_ctrl,round_1_fighter_2_total_kd,round_1_fighter_2_total_rev,round_1_fighter_2_total_sig_str,round_1_fighter_2_total_sig_str_prcnt,round_1_fighter_2_total_sub_att,round_1_fighter_2_total_td,round_1_fighter_2_total_td_prcnt,round_1_fighter_2_total_total_str,round_2_fighter_1_sig_str_body,round_2_fighter_1_sig_str_clinch,round_2_fighter_1_sig_str_distance,round_2_fighter_1_sig_str_ground,round_2_fighter_1_sig_str_head,round_2_fighter_1_sig_str_leg,round_2_fighter_1_sig_str_sig_str,round_2_fighter_1_sig_str_sig_str_prcnt,round_2_fighter_1_total_ctrl,round_2_fighter_1_total_kd,round_2_fighter_1_total_rev,round_2_fighter_1_total_sig_str,round_2_fighter_1_total_sig_str_prcnt,round_2_fighter_1_total_sub_att,round_2_fighter_1_total_td,round_2_fighter_1_total_td_prcnt,round_2_fighter_1_total_total_str,round_2_fighter_2_sig_str_body,round_2_fighter_2_sig_str_clinch,round_2_fighter_2_sig_str_distance,round_2_fighter_2_sig_str_ground,round_2_fighter_2_sig_str_head,round_2_fighter_2_sig_str_leg,round_2_fighter_2_sig_str_sig_str,round_2_fighter_2_sig_str_sig_str_prcnt,round_2_fighter_2_total_ctrl,round_2_fighter_2_total_kd,round_2_fighter_2_total_rev,round_2_fighter_2_total_sig_str,round_2_fighter_2_total_sig_str_prcnt,round_2_fighter_2_total_sub_att,round_2_fighter_2_total_td,round_2_fighter_2_total_td_prcnt,round_2_fighter_2_total_total_str,round_3_fighter_1_sig_str_body,round_3_fighter_1_sig_str_clinch,round_3_fighter_1_sig_str_distance,round_3_fighter_1_sig_str_ground,round_3_fighter_1_sig_str_head,round_3_fighter_1_sig_str_leg,round_3_fighter_1_sig_str_sig_str,round_3_fighter_1_sig_str_sig_str_prcnt,round_3_fighter_1_total_ctrl,round_3_fighter_1_total_kd,round_3_fighter_1_total_rev,round_3_fighter_1_total_sig_str,round_3_fighter_1_total_sig_str_prcnt,round_3_fighter_1_total_sub_att,round_3_fighter_1_total_td,round_3_fighter_1_total_td_prcnt,round_3_fighter_1_total_total_str,round_3_fighter_2_sig_str_body,round_3_fighter_2_sig_str_clinch,round_3_fighter_2_sig_str_distance,round_3_fighter_2_sig_str_ground,round_3_fighter_2_sig_str_head,round_3_fighter_2_sig_str_leg,round_3_fighter_2_sig_str_sig_str,round_3_fighter_2_sig_str_sig_str_prcnt,round_3_fighter_2_total_ctrl,round_3_fighter_

In [13]:
import pandas as pd
import numpy as np
import re

def clean_ufc_dataframe_v2(df):
    """
    Optimized UFC cleaning function.
    Fixes fragmentation warnings and corrects scheduled_rounds logic.
    """
    # Create a copy to work on and avoid SettingWithCopy warnings
    df_clean = df.copy()

    # ==========================================
    # HELPER FUNCTIONS
    # ==========================================
    def split_landed_attempted(val):
        """Splits '17 of 26' into (17, 26)."""
        if not isinstance(val, str) or ' of ' not in val:
            return np.nan, np.nan
        try:
            landed, attempted = val.split(' of ')
            return int(landed), int(attempted)
        except ValueError:
            return np.nan, np.nan

    def clean_time_to_seconds(val):
        """Converts 'M:SS' to integer seconds."""
        if pd.isna(val) or str(val).strip() in ['--', '---', 'nan']:
            return 0
        try:
            parts = str(val).split(':')
            if len(parts) == 2:
                return int(parts[0]) * 60 + int(parts[1])
            return 0
        except ValueError:
            return 0

    def clean_percentage(val):
        """Converts '65%' to 0.65."""
        if pd.isna(val) or str(val).strip() == '---':
            return 0.0
        if isinstance(val, str) and '%' in val:
            return float(val.strip('%')) / 100
        return float(val)

    def extract_scheduled_from_format(val):
        """
        Takes '5 Rnd (5-5-5-5-5)' and returns 5.
        Logic: Take first character of the string.
        """
        if pd.isna(val):
            return 3 # Default fallback
        s_val = str(val).strip()
        if not s_val:
            return 3
        try:
            # Grab just the first character (e.g., "5" from "5 Rnd...")
            return int(s_val[0])
        except ValueError:
            return 3

    # ==========================================
    # 1. THE "SPLIT" GROUP (Optimized - No Warnings)
    # ==========================================
    target_keywords = ['sig_str', 'total_str', 'td', 'head', 'body', 'leg', 'distance', 'clinch', 'ground']
    
    # Identify columns to split
    cols_to_split = [
        c for c in df_clean.columns 
        if any(k in c for k in target_keywords) 
        and 'prcnt' not in c 
        and 'ctrl' not in c
        and df_clean[c].dtype == object 
        and df_clean[c].astype(str).str.contains(' of ').any()
    ]

    # --- OPTIMIZATION: Collect data first, concat once ---
    new_data = {}
    
    for col in cols_to_split:
        # Run logic
        split_values = df_clean[col].apply(split_landed_attempted)
        
        # Store in dict
        new_data[f'{col}_landed'] = split_values.apply(lambda x: x[0])
        new_data[f'{col}_attempted'] = split_values.apply(lambda x: x[1])

    # Batch add all new columns
    if new_data:
        new_cols_df = pd.DataFrame(new_data, index=df_clean.index)
        df_clean = pd.concat([df_clean, new_cols_df], axis=1)
    
    # Batch drop old columns
    df_clean.drop(columns=cols_to_split, inplace=True)

    # ==========================================
    # 2. THE PERCENTAGE GROUP
    # ==========================================
    pct_cols = [c for c in df_clean.columns if 'prcnt' in c]
    for col in pct_cols:
        df_clean[col] = df_clean[col].apply(clean_percentage)

    # ==========================================
    # 3. THE TIME GROUP
    # ==========================================
    # Note: We do NOT include meta_time here as we are dropping it later
    time_cols = ['time'] + [c for c in df_clean.columns if 'ctrl' in c]
    
    for col in time_cols:
        if col in df_clean.columns:
            df_clean[col] = df_clean[col].apply(clean_time_to_seconds)

    # ==========================================
    # 4. STRUCTURE & META DATA
    # ==========================================
    
    # Fix: Scheduled Rounds from 'meta_time_format'
    if 'meta_time_format' in df_clean.columns:
        df_clean['scheduled_rounds'] = df_clean['meta_time_format'].apply(extract_scheduled_from_format)
    
    # Drop redundant columns (Added meta_time to this list)
    redundant_cols = ['winner', 'meta_method', 'meta_time_format', 'meta_referee', 'meta_round', 'meta_time']
    df_clean.drop(columns=[c for c in redundant_cols if c in df_clean.columns], inplace=True)

    # ==========================================
    # 6. DATE HANDLING
    # ==========================================
    if 'event_date' in df_clean.columns:
        df_clean['event_date'] = pd.to_datetime(df_clean['event_date'])

    # ==========================================
    # NEW: TOTAL FIGHT TIME CALCULATION
    # ==========================================
    # Formula: (round - 1) * 5 minutes + time_in_last_round
    
    # Ensure round is numeric
    df_clean['round'] = pd.to_numeric(df_clean['round'], errors='coerce').fillna(1)
    
    # 'time' is already converted to seconds in Step 3
    df_clean['total_fight_time_secs'] = ((df_clean['round'] - 1) * 5 * 60) + df_clean['time']

    # ==========================================
    # 5. HANDLING NaNs (Zero Fill for R2-R5)
    # ==========================================
    # Target columns starting with round_2, round_3, round_4, round_5
    round_cols = [c for c in df_clean.columns if re.match(r'round_[2-5]_', c)]
    
    df_clean[round_cols] = df_clean[round_cols].fillna(0)

    # Return a defragmented copy
    return df_clean.copy()

# === USAGE ===
df_cleaned = clean_ufc_dataframe_v2(df_formatted)
df_cleaned.head()

,event_date,fighter_1,fighter_1_match_ctrl,fighter_1_match_kd,fighter_1_match_rev,fighter_1_match_sig_str_prcnt,fighter_1_match_sub_att,fighter_1_match_td_prcnt,fighter_1_sig_str_sig_str_prcnt,fighter_2,fighter_2_match_ctrl,fighter_2_match_kd,fighter_2_match_rev,fighter_2_match_sig_str_prcnt,fighter_2_match_sub_att,fighter_2_match_td_prcnt,fighter_2_sig_str_sig_str_prcnt,meta_first_fighter_won,method,round,round_1_fighter_1_sig_str_sig_str_prcnt,round_1_fighter_1_total_ctrl,round_1_fighter_1_total_kd,round_1_fighter_1_total_rev,round_1_fighter_1_total_sig_str_prcnt,round_1_fighter_1_total_sub_att,round_1_fighter_1_total_td_prcnt,round_1_fighter_2_sig_str_sig_str_prcnt,round_1_fighter_2_total_ctrl,round_1_fighter_2_total_kd,round_1_fighter_2_total_rev,round_1_fighter_2_total_sig_str_prcnt,round_1_fighter_2_total_sub_att,round_1_fighter_2_total_td_prcnt,round_2_fighter_1_sig_str_sig_str_prcnt,round_2_fighter_1_total_ctrl,round_2_fighter_1_total_kd,round_2_fighter_1_total_rev,round_2_fighter_1_total_sig_str_prcnt,round_2_fighter_1_total_sub_att,round_2_fighter_1_total_td_prcnt,round_2_fighter_2_sig_str_sig_str_prcnt,round_2_fighter_2_total_ctrl,round_2_fighter_2_total_kd,round_2_fighter_2_total_rev,round_2_fighter_2_total_sig_str_prcnt,round_2_fighter_2_total_sub_att,round_2_fighter_2_total_td_prcnt,round_3_fighter_1_sig_str_sig_str_prcnt,round_3_fighter_1_total_ctrl,round_3_fighter_1_total_kd,round_3_fighter_1_total_rev,round_3_fighter_1_total_sig_str_prcnt,round_3_fighter_1_total_sub_att,round_3_fighter_1_total_td_prcnt,round_3_fighter_2_sig_str_sig_str_prcnt,round_3_fighter_2_total_ctrl,round_3_fighter_2_total_kd,round_3_fighter_2_total_rev,round_3_fighter_2_total_sig_str_prcnt,round_3_fighter_2_total_sub_att,round_3_fighter_2_total_td_prcnt,round_4_fighter_1_sig_str_sig_str_prcnt,round_4_fighter_1_total_ctrl,round_4_fighter_1_total_kd,round_4_fighter_1_total_rev,round_4_fighter_1_total_sig_str_prcnt,round_4_fighter_1_total_sub_att,round_4_fighter_1_total_td_prcnt,round_4_fighter_2_sig_str_sig_str_prcnt,round_4_fighter_2_total_ctrl,round_4_fighter_2_total_kd,round_4_fighter_2_total_rev,round_4_fighter_2_total_sig_str_prcnt,round_4_fighter_2_total_sub_att,round_4_fighter_2_total_td_prcnt,round_5_fighter_1_sig_str_sig_str_prcnt,round_5_fighter_1_total_ctrl,round_5_fighter_1_total_kd,round_5_fighter_1_total_rev,round_5_fighter_1_total_sig_str_prcnt,round_5_fighter_1_total_sub_att,round_5_fighter_1_total_td_prcnt,round_5_fighter_2_sig_str_sig_str_prcnt,round_5_fighter_2_total_ctrl,round_5_fighter_2_total_kd,round_5_fighter_2_total_rev,round_5_fighter_2_total_sig_str_prcnt,round_5_fighter_2_total_sub_att,round_5_fighter_2_total_td_prcnt,time,fighter_1_match_sig_str_landed,fighter_1_match_sig_str_attempted,fighter_1_match_td_landed,fighter_1_match_td_attempted,fighter_1_match_total_str_landed,fighter_1_match_total_str_attempted,fighter_1_sig_str_body_landed,fighter_1_sig_str_body_attempted,fighter_1_sig_str_clinch_landed,fighter_1_sig_str_clinch_attempted,fighter_1_sig_str_distance_landed,fighter_1_sig_str_distance_attempted,fighter_1_sig_str_ground_landed,fighter_1_sig_str_ground_attempted,fighter_1_sig_str_head_landed,fighter_1_sig_str_head_attempted,fighter_1_sig_str_leg_landed,fighter_1_sig_str_leg_attempted,fighter_1_sig_str_sig_str_landed,fighter_1_sig_str_sig_str_attempted,fighter_2_match_sig_str_landed,fighter_2_match_sig_str_attempted,fighter_2_match_td_landed,fighter_2_match_td_attempted,fighter_2_match_total_str_landed,fighter_2_match_total_str_attempted,fighter_2_sig_str_body_landed,fighter_2_sig_str_body_attempted,fighter_2_sig_str_clinch_landed,fighter_2_sig_str_clinch_attempted,fighter_2_sig_str_distance_landed,fighter_2_sig_str_distance_attempted,fighter_2_sig_str_ground_landed,fighter_2_sig_str_ground_attempted,fighter_2_sig_str_head_landed,fighter_2_sig_str_head_attempted,fighter_2_sig_str_leg_landed,fighter_2_sig_str_leg_attempted,fighter_2_sig_str_sig_str_landed,fighter_2_sig_str_sig_str_attempted,round_1_fighter

In [14]:
import pandas as pd

def map_fighter_dob_and_age(df_matches, fighters_csv_path):
    """
    Maps fighter DOBs, calculates Age, and then removes DOB.
    """
    # 1. Load & Clean Fighters Data
    df_fighters = pd.read_csv(fighters_csv_path)
    df_fighters['dob'] = pd.to_datetime(df_fighters['dob'], errors='coerce')
    
    # Drop duplicates to prevent row explosion
    df_fighters = df_fighters.drop_duplicates(subset=['name'])
    
    # Keep only relevant columns
    fighter_lookup = df_fighters[['name', 'dob']]
    
    # 2. Merge for Fighter 1
    df_merged = df_matches.merge(
        fighter_lookup, 
        left_on='fighter_1', 
        right_on='name', 
        how='left'
    )
    df_merged.rename(columns={'dob': 'fighter_1_dob'}, inplace=True)
    df_merged.drop(columns=['name'], inplace=True)
    
    # 3. Merge for Fighter 2
    df_merged = df_merged.merge(
        fighter_lookup, 
        left_on='fighter_2', 
        right_on='name', 
        how='left'
    )
    df_merged.rename(columns={'dob': 'fighter_2_dob'}, inplace=True)
    df_merged.drop(columns=['name'], inplace=True)
    
    # 4. Calculate Age & DROP DOB
    if 'event_date' in df_merged.columns:
        # Calculate Age
        df_merged['fighter_1_age'] = (df_merged['event_date'] - df_merged['fighter_1_dob']).dt.days / 365.25
        df_merged['fighter_2_age'] = (df_merged['event_date'] - df_merged['fighter_2_dob']).dt.days / 365.25
        
        # --- DROP THE DOB COLUMNS HERE ---
        df_merged.drop(columns=['fighter_1_dob', 'fighter_2_dob'], inplace=True)

    return df_merged

# === USAGE ===
df_final = map_fighter_dob_and_age(df_cleaned, 'ufc_fighters.csv')

df_final.head()

,event_date,fighter_1,fighter_1_match_ctrl,fighter_1_match_kd,fighter_1_match_rev,fighter_1_match_sig_str_prcnt,fighter_1_match_sub_att,fighter_1_match_td_prcnt,fighter_1_sig_str_sig_str_prcnt,fighter_2,fighter_2_match_ctrl,fighter_2_match_kd,fighter_2_match_rev,fighter_2_match_sig_str_prcnt,fighter_2_match_sub_att,fighter_2_match_td_prcnt,fighter_2_sig_str_sig_str_prcnt,meta_first_fighter_won,method,round,round_1_fighter_1_sig_str_sig_str_prcnt,round_1_fighter_1_total_ctrl,round_1_fighter_1_total_kd,round_1_fighter_1_total_rev,round_1_fighter_1_total_sig_str_prcnt,round_1_fighter_1_total_sub_att,round_1_fighter_1_total_td_prcnt,round_1_fighter_2_sig_str_sig_str_prcnt,round_1_fighter_2_total_ctrl,round_1_fighter_2_total_kd,round_1_fighter_2_total_rev,round_1_fighter_2_total_sig_str_prcnt,round_1_fighter_2_total_sub_att,round_1_fighter_2_total_td_prcnt,round_2_fighter_1_sig_str_sig_str_prcnt,round_2_fighter_1_total_ctrl,round_2_fighter_1_total_kd,round_2_fighter_1_total_rev,round_2_fighter_1_total_sig_str_prcnt,round_2_fighter_1_total_sub_att,round_2_fighter_1_total_td_prcnt,round_2_fighter_2_sig_str_sig_str_prcnt,round_2_fighter_2_total_ctrl,round_2_fighter_2_total_kd,round_2_fighter_2_total_rev,round_2_fighter_2_total_sig_str_prcnt,round_2_fighter_2_total_sub_att,round_2_fighter_2_total_td_prcnt,round_3_fighter_1_sig_str_sig_str_prcnt,round_3_fighter_1_total_ctrl,round_3_fighter_1_total_kd,round_3_fighter_1_total_rev,round_3_fighter_1_total_sig_str_prcnt,round_3_fighter_1_total_sub_att,round_3_fighter_1_total_td_prcnt,round_3_fighter_2_sig_str_sig_str_prcnt,round_3_fighter_2_total_ctrl,round_3_fighter_2_total_kd,round_3_fighter_2_total_rev,round_3_fighter_2_total_sig_str_prcnt,round_3_fighter_2_total_sub_att,round_3_fighter_2_total_td_prcnt,round_4_fighter_1_sig_str_sig_str_prcnt,round_4_fighter_1_total_ctrl,round_4_fighter_1_total_kd,round_4_fighter_1_total_rev,round_4_fighter_1_total_sig_str_prcnt,round_4_fighter_1_total_sub_att,round_4_fighter_1_total_td_prcnt,round_4_fighter_2_sig_str_sig_str_prcnt,round_4_fighter_2_total_ctrl,round_4_fighter_2_total_kd,round_4_fighter_2_total_rev,round_4_fighter_2_total_sig_str_prcnt,round_4_fighter_2_total_sub_att,round_4_fighter_2_total_td_prcnt,round_5_fighter_1_sig_str_sig_str_prcnt,round_5_fighter_1_total_ctrl,round_5_fighter_1_total_kd,round_5_fighter_1_total_rev,round_5_fighter_1_total_sig_str_prcnt,round_5_fighter_1_total_sub_att,round_5_fighter_1_total_td_prcnt,round_5_fighter_2_sig_str_sig_str_prcnt,round_5_fighter_2_total_ctrl,round_5_fighter_2_total_kd,round_5_fighter_2_total_rev,round_5_fighter_2_total_sig_str_prcnt,round_5_fighter_2_total_sub_att,round_5_fighter_2_total_td_prcnt,time,fighter_1_match_sig_str_landed,fighter_1_match_sig_str_attempted,fighter_1_match_td_landed,fighter_1_match_td_attempted,fighter_1_match_total_str_landed,fighter_1_match_total_str_attempted,fighter_1_sig_str_body_landed,fighter_1_sig_str_body_attempted,fighter_1_sig_str_clinch_landed,fighter_1_sig_str_clinch_attempted,fighter_1_sig_str_distance_landed,fighter_1_sig_str_distance_attempted,fighter_1_sig_str_ground_landed,fighter_1_sig_str_ground_attempted,fighter_1_sig_str_head_landed,fighter_1_sig_str_head_attempted,fighter_1_sig_str_leg_landed,fighter_1_sig_str_leg_attempted,fighter_1_sig_str_sig_str_landed,fighter_1_sig_str_sig_str_attempted,fighter_2_match_sig_str_landed,fighter_2_match_sig_str_attempted,fighter_2_match_td_landed,fighter_2_match_td_attempted,fighter_2_match_total_str_landed,fighter_2_match_total_str_attempted,fighter_2_sig_str_body_landed,fighter_2_sig_str_body_attempted,fighter_2_sig_str_clinch_landed,fighter_2_sig_str_clinch_attempted,fighter_2_sig_str_distance_landed,fighter_2_sig_str_distance_attempted,fighter_2_sig_str_ground_landed,fighter_2_sig_str_ground_attempted,fighter_2_sig_str_head_landed,fighter_2_sig_str_head_attempted,fighter_2_sig_str_leg_landed,fighter_2_sig_str_leg_attempted,fighter_2_sig_str_sig_str_landed,fighter_2_sig_str_sig_str_attempted,round_1_fighter

In [ ]:
import pandas as pd
import numpy as np

def engineer_ufc_features_v2(df_clean):
    """
    Advanced Feature Engineering with:
    1. Conditional EMA (Pauses history for rounds that didn't happen).
    2. Full Round 1-5 support.
    3. Fight Time Normalization.
    """
    print("--- Starting Advanced Feature Engineering ---")
    
    # ==========================================
    # 1. RESHAPE TO LONG FORMAT & MAP DEFENSE
    # ==========================================
    cross_maps = [
        ('sig_str_absorbed', 'fighter_2_match_sig_str_landed', 'fighter_1_match_sig_str_landed'),
        ('td_absorbed',      'fighter_2_match_td_landed',      'fighter_1_match_td_landed'),
        ('ctrl_absorbed',    'fighter_2_match_ctrl',           'fighter_1_match_ctrl'),
    ]

    # Process Fighter 1
    f1_cols = [c for c in df_clean.columns if c.startswith('fighter_1_')]
    f1_df = df_clean[['event_date', 'fighter_1', 'fighter_2', 'total_fight_time_secs', 'scheduled_rounds', 'meta_first_fighter_won'] + f1_cols].copy()
    f1_df.columns = [c.replace('fighter_1_', '') for c in f1_df.columns]
    f1_df.rename(columns={'fighter_1': 'fighter_name', 'fighter_2': 'opponent_name'}, inplace=True)
    f1_df['is_winner'] = df_clean['meta_first_fighter_won'].astype(int)
    for new, f1, f2 in cross_maps: f1_df[new] = df_clean[f1]

    # Process Fighter 2
    f2_cols = [c for c in df_clean.columns if c.startswith('fighter_2_')]
    f2_df = df_clean[['event_date', 'fighter_2', 'fighter_1', 'total_fight_time_secs', 'scheduled_rounds', 'meta_first_fighter_won'] + f2_cols].copy()
    f2_df.columns = [c.replace('fighter_2_', '') for c in f2_df.columns]
    f2_df.rename(columns={'fighter_2': 'fighter_name', 'fighter_1': 'opponent_name'}, inplace=True)
    f2_df['is_winner'] = (~df_clean['meta_first_fighter_won']).astype(int)
    for new, f1, f2 in cross_maps: f2_df[new] = df_clean[f2]

    # Stack
    long_df = pd.concat([f1_df, f2_df], axis=0)
    long_df['event_date'] = pd.to_datetime(long_df['event_date'])
    long_df = long_df.sort_values(['fighter_name', 'event_date']).reset_index(drop=True)

    # ==========================================
    # 2. CALCULATE RATES (The Normalization Fix)
    # ==========================================
    # We use Per Minute (PM) for totals to solve the "longer fight = higher stats" bias
    time_min = long_df['total_fight_time_secs'] / 60
    time_min = time_min.replace(0, 1)
    
    long_df['slpm'] = long_df['match_sig_str_landed'] / time_min
    long_df['sapm'] = long_df['sig_str_absorbed'] / time_min
    long_df['td_avg'] = (long_df['match_td_landed'] / time_min) * 15
    
    # New Feature: Average Fight Time (Are they a finisher or grinder?)
    # We will EMA this later.
    long_df['fight_time_stat'] = long_df['total_fight_time_secs']

    # ==========================================
    # 3. PREPARE ROUNDS 1-5 (The "Conditional" Logic)
    # ==========================================
    # We want to capture Round 1-5 stats.
    # CRITICAL STEP: Replace 0s with NaN for rounds that didn't happen.
    # This prevents a R1 KO from dragging down the R5 Average.
    
    round_cols_to_smooth = []
    
    # Loop through all 5 rounds
    for r in range(1, 6):
        # Define the columns we want history for
        target_metrics = [
            f'round_{r}_sig_str_landed',
            f'round_{r}_sig_str_attempted',
            f'round_{r}_total_td_landed',
            f'round_{r}_total_td_attempted'
        ]
        
        for col in target_metrics:
            if col in long_df.columns:
                # TRICK: Identify rows where this round actually happened.
                # If 'sig_str_attempted' is 0 for a round, likely the round didn't happen 
                # (or they did literally nothing, but usually it means the fight ended).
                # We set it to NaN so .ewm() skips it.
                long_df[col] = long_df[col].replace(0, np.nan)
                round_cols_to_smooth.append(col)

    # ==========================================
    # 4. APPLY ROLLING EMA (History Generator)
    # ==========================================
    # Combined list: Rates + Fight Time + Round Specifics
    cols_to_roll = ['slpm', 'sapm', 'td_avg', 'fight_time_stat'] + round_cols_to_smooth
    
    print(f"Generating history for {len(cols_to_roll)} features...")

    # Calculate EMA with Shift=1 (No data leakage)
    # Pandas .ewm(ignore_na=False) is default, which means if it sees NaN, it holds the previous value.
    ema_df = long_df.groupby('fighter_name')[cols_to_roll].transform(
        lambda x: x.ewm(span=3, min_periods=1).mean().shift(1)
    )
    ema_df.columns = [f"{c}_ema" for c in ema_df.columns]
    
    # Merge back
    long_df = pd.concat([long_df, ema_df], axis=1)

    # ==========================================
    # 5. PACING RATIOS (Cardio Decay)
    # ==========================================
    # Now we have "True" EMAs. If R3 EMA is NaN, it means they have NEVER been to Round 3.
    # We handle that fillna(1) so ratio is 1.0 (Neutral).
    
    # R3 vs R1 (Main Event Cardio)
    if 'round_3_sig_str_landed_ema' in long_df.columns:
        long_df['cardio_decay_r3'] = long_df['round_3_sig_str_landed_ema'] / long_df['round_1_sig_str_landed_ema'].replace(0, 1)
        long_df['cardio_decay_r3'] = long_df['cardio_decay_r3'].fillna(1) # Neutral if no history

    # R5 vs R1 (Championship Cardio)
    if 'round_5_sig_str_landed_ema' in long_df.columns:
        long_df['cardio_decay_r5'] = long_df['round_5_sig_str_landed_ema'] / long_df['round_1_sig_str_landed_ema'].replace(0, 1)
        long_df['cardio_decay_r5'] = long_df['cardio_decay_r5'].fillna(1)

    # ==========================================
    # 6. RESHAPE TO MATCH & DIFF
    # ==========================================
    feature_cols = [c for c in long_df.columns if c.endswith('_ema') or 'cardio_decay' in c]
    lookup = long_df[['event_date', 'fighter_name'] + feature_cols]
    
    # Merge P1
    df_final = df_clean.merge(lookup, left_on=['event_date', 'fighter_1'], right_on=['event_date', 'fighter_name'], how='left').drop(columns=['fighter_name'])
    df_final.rename(columns={c: f"p1_{c}" for c in feature_cols}, inplace=True)
    
    # Merge P2
    df_final = df_final.merge(lookup, left_on=['event_date', 'fighter_2'], right_on=['event_date', 'fighter_name'], how='left').drop(columns=['fighter_name'])
    df_final.rename(columns={c: f"p2_{c}" for c in feature_cols}, inplace=True)
    
    # Diffs
    for col in feature_cols:
        p1 = f"p1_{col}"
        p2 = f"p2_{col}"
        if p1 in df_final.columns and p2 in df_final.columns:
            df_final[f"{col}_diff"] = df_final[p1] - df_final[p2]
            
    return df_final

df_features = engineer_ufc_features_v2(df_final)